# Object Tracking
This practical introduces **object tracking** as the problem of maintaining object identity over time as objects and cameras move, a core capability in many real-world vision systems such as autonomous driving, robotics, video surveillance and human-computer interaction. Tracking requires **detecting objects** in each frame and associating them across time while they undergo motion, interact with other objects, enter or leave the field of view and experience temporary occlusion.

You will explore this problem in two stages.

In Part I, you will use **Grounding DINO** as an open-vocabulary detection module for tracking in images and videos. You will begin with text-conditioned object detection on still images to understand how prompt design, multiple object queries and attribute refinement define tracking targets. You will then apply the same detection process frame by frame to video sequences, observing how object locations and confidence scores evolve over time.

In Part II, you will build a tracking-by-detection system using **Intersection-over-Union (IoU)**-based association. Starting from the per-frame detections produced by Grounding DINO, you will associate detections across consecutive frames to form object tracks, handle object appearance and disappearance and analyse the resulting trajectories.

Together, these stages provide a practical introduction to open-vocabulary object tracking, showing how **language-conditioned detectors** can be used to specify tracking targets and how temporal association transforms per-frame detections into reasonable **object trajectories**.

*Before you start*

This practical runs both in Google Colab and in a local Jupyter Notebook installation. If you are using Colab, select a GPU runtime via *Runtime → Change runtime type*.

## 0. Preliminaries

The single cell below performs all of the setup for this practical: it reports the compute device, downloads and verifies the image and video data, downloads the Grounding DINO weights, and installs Grounding DINO itself.

Everything is written to paths **relative to this notebook** — `data/`, `weights/` and `GroundingDINO/` — so nothing depends on the notebook running in any particular directory. The cell is safe to re-run: downloads and the repository clone are skipped if they are already present.

Outside Colab you will need `git`, `wget`, `tar`, `unzip` and `ffmpeg` on your `PATH`, and a working PyTorch installation (see [pytorch.org](https://pytorch.org/get-started/locally/); the CPU-only build is sufficient).

In [ ]:
import importlib
import os
import site

import torch
from torch.utils.cpp_extension import CUDA_HOME

print("PyTorch version  :", torch.__version__)
print("Built with CUDA  :", torch.version.cuda)   # None => CPU-only build
print("CUDA available   :", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device     :", DEVICE)
if DEVICE == "cpu":
    print("Running on the CPU: everything works, but the video cells are slow.")

os.makedirs("data", exist_ok=True)
os.makedirs("weights", exist_ok=True)

!wget --no-verbose --no-clobber -P data https://www.robots.ox.ac.uk/~vgg/share/practical-instance-recognition-2018a-data-only.tar.gz
!echo "5b2d65c2c8cfc030486323a484b840f3  data/practical-instance-recognition-2018a-data-only.tar.gz" | md5sum --check || exit 1
!tar -xzf data/practical-instance-recognition-2018a-data-only.tar.gz -C data --strip-components=2 practical-instance-recognition-2018a/data/oxbuild_lite

!wget --no-verbose --no-clobber -P data https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-test-dev-480p.zip
!echo "449209d75d3cabd1c021f233dca1e9b0  data/DAVIS-2017-test-dev-480p.zip" | md5sum --check || exit 1
!unzip -q -o data/DAVIS-2017-test-dev-480p.zip -d data

!ffmpeg -y -loglevel error \
  -framerate 24 \
  -start_number 0 \
  -i data/DAVIS/JPEGImages/480p/tennis-vest/%05d.jpg \
  -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
  -c:v libx264 -pix_fmt yuv420p \
  data/tennis-vest.mp4

!ffmpeg -y -loglevel error \
  -framerate 24 \
  -start_number 0 \
  -i data/DAVIS/JPEGImages/480p/horsejump-stick/%05d.jpg \
  -vf "scale=trunc(iw/2)*2:trunc(ih/2)*2" \
  -c:v libx264 -pix_fmt yuv420p \
  data/horsejump-stick.mp4

!wget --no-verbose --no-clobber -P weights https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!echo "075ebfa7242d913f38cb051fe1a128a2  weights/groundingdino_swint_ogc.pth" | md5sum --check || exit 1

GROUNDING_DINO_COMMIT = "856dde20aee659246248e20734ef9ba5214f5e44"

if not os.path.isdir("GroundingDINO"):
    !git clone https://github.com/IDEA-Research/GroundingDINO.git
!git -C GroundingDINO checkout --quiet {GROUNDING_DINO_COMMIT}

# Grounding DINO ships a hand-written CUDA kernel for multi-scale deformable
# attention. Its source calls `value.type()`, which modern versions of ATen no
# longer accept where a `ScalarType` is expected, so the kernel fails to
# compile. The two substitutions below fix that (the second repairs the
# `value.type().is_cuda()` calls that the first one would otherwise mangle);
# both forms are valid on older PyTorch too. See
# https://github.com/IDEA-Research/Grounded-Segment-Anything/issues/550#issuecomment-2760092141
#
# The kernel is only compiled when a CUDA toolkit *and* a GPU are present --
# see `get_extensions()` in GroundingDINO/setup.py -- so there is nothing to
# patch otherwise, and we check that the patch is actually still needed before
# applying it.
MS_DEFORM_ATTN_CU = os.path.join(
    "GroundingDINO", "groundingdino", "models", "GroundingDINO",
    "csrc", "MsDeformAttn", "ms_deform_attn_cuda.cu")

if CUDA_HOME is None or not torch.cuda.is_available():
    print("No CUDA toolkit and/or no GPU: the CUDA extension will not be built,")
    print("so ms_deform_attn_cuda.cu is left untouched. Grounding DINO will use")
    print("its pure-PyTorch deformable-attention fallback.")
elif "value.type()" in open(MS_DEFORM_ATTN_CU).read():
    print("CUDA toolkit at %s, PyTorch built with CUDA %s: patching %s"
          % (CUDA_HOME, torch.version.cuda, MS_DEFORM_ATTN_CU))
    !sed -i 's/value\.type()/value.scalar_type()/g; s/value\.scalar_type()\.is_cuda()/value.is_cuda()/g' {MS_DEFORM_ATTN_CU}
else:
    print("ms_deform_attn_cuda.cu needs no patching.")


%pip install -q setuptools wheel imageio imageio-ffmpeg matplotlib "transformers<5"
%pip install -q --no-build-isolation -e GroundingDINO

importlib.reload(site)
importlib.invalidate_caches()

CONFIG_PATH = os.path.join(
    "GroundingDINO", "groundingdino", "config", "GroundingDINO_SwinT_OGC.py")
print(CONFIG_PATH, "; exist:", os.path.isfile(CONFIG_PATH))

import cv2
import groundingdino.datasets.transforms as T
import imageio.v2 as imageio
import numpy as np
import supervision as sv
from groundingdino.util.inference import (
    annotate,
    load_image,
    load_model,
    predict,
)
from IPython.display import Video
from matplotlib import pyplot as plt
from PIL import Image, ImageDraw, ImageFont

%matplotlib inline

## Part I: Open-Vocabulary Detection with Grounding DINO
In this part, you will use **Grounding DINO** to perform **open-vocabulary object detection**, which will serve as the detection backbone for the tracking system developed later in the practical. You will begin by applying Grounding DINO to still images to understand how **text prompts** control the detector's output. You will experiment with single-object prompts and refined prompts that include object attributes.

You will then apply the same detection process to video sequences, running Grounding DINO independently on each frame. By visualising detections over time, you will observe how object locations evolve as objects move and interact within the scene. This frame-wise detection output will form the input to the tracking-by-detection pipeline in Part II, where temporal association will be introduced to maintain object identity across frames.
### Learning Outcomes

By the end of Part I, you should be able to:
* Explain the principle of open-vocabulary object detection and how it differs from fixed-category detection.
* Use Grounding DINO to perform text-conditioned object detection on still images.
* Design and refine natural-language prompts to control which objects are detected.
* Apply open-vocabulary detection frame by frame to a video sequence.
* Interpret how object motion over time is reflected in sequences of per-frame detections.
* Prepare detection outputs suitable for tracking-by-detection, including bounding boxes and confidence scores.

**Grounding DINO** is trained to **align visual features with natural language descriptions**, enabling object detection conditioned on text prompts. It builds on a pretrained vision backbone and a language encoder, and is trained with **supervised image-text data** in which textual phrases are explicitly grounded to object bounding boxes rather than treated as fixed class labels. During training, the model learns region-phrase correspondence through a combination of DETR-style set-based **bounding box regression** and **cross-modal alignment objectives**. This allows Grounding DINO to localise objects described in free-form language and to generalise beyond the closed set of categories seen during training.

Before performing any detection, we first load the Grounding DINO model and its pretrained weights.

In [ ]:
WEIGHTS_PATH = os.path.join("weights", "groundingdino_swint_ogc.pth")
print(WEIGHTS_PATH, "; exist:", os.path.isfile(WEIGHTS_PATH))
model = load_model(CONFIG_PATH, WEIGHTS_PATH, device=DEVICE)

We now apply Grounding DINO to a single image to perform open-vocabulary object detection using a text prompt. The detection target is specified by the variable `TEXT_PROMPT`, which defines the object category to be localised in the image. In this example, the prompt **"person"** is used to detect image regions corresponding to people.


In [ ]:
image_path = "data/oxbuild_lite/cornmarket_000027.jpg"

TEXT_PROMPT = "person"
BOX_TRESHOLD = 0.35
TEXT_TRESHOLD = 0.25

image_source, image = load_image(image_path)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)

sv.plot_image(annotated_frame, (16, 16))

> *Question 1: Grounding DINO uses two thresholds to control its detection output. How do the box threshold and the text threshold influence the number and quality of detections and what effect would increasing or decreasing each threshold have on the results? What implications might this have when the detector is applied frame by frame in a video sequence?*

The predict function returns three key outputs that describe the detections produced for a given text prompt:
* **boxes**:
These are the predicted bounding boxes for the detected objects. Each box specifies the spatial extent of a detected region in the image, typically represented as coordinates relative to the image size. The boxes indicate where in the image the model believes an object matching the text prompt is located.
* **logits**:
These values represent the confidence scores associated with each predicted bounding box. They quantify how strongly the model believes that a given box corresponds to the text prompt. Higher logits indicate a stronger alignment between the image region and the prompt.
* **phrases**:
These are the text phrases associated with each detection. When multiple prompts are provided, this indicates which phrase each bounding box has been matched to. In this case, with a single prompt, all returned phrases correspond to “person”.

Together, these outputs define the detection hypotheses for the image: the boxes specify the spatial locations, the logits indicate confidence, and the phrases link each detection back to the text prompt.

We now examine an individual detection in more detail by converting its predicted bounding box from the model’s normalised, centre-based representation into image pixel coordinates. This allows us to directly relate the numerical output of the detector to a concrete region in the image.

In [ ]:
index = 0
H, W, C = image_source.shape
box_i, score_i, phrase_i = boxes[index], logits[index], phrases[0]

# box_i (x_c, y_c, w, h)

def convert_box_to_image_coordinates(box, H,W):
  x_c, y_c, bw, bh = box.numpy()
  x_c *= W
  y_c *= H
  bw  *= W
  bh  *= H

  x_min = x_c - bw / 2
  y_min = y_c - bh / 2
  x_max = x_c + bw / 2
  y_max = y_c + bh / 2

  x_min = max(0, min(W - 1, x_min))
  y_min = max(0, min(H - 1, y_min))
  x_max = max(0, min(W - 1, x_max))
  y_max = max(0, min(H - 1, y_max))
  return int(x_min), int(y_min), int(x_max), int(y_max)

x_min, y_min, x_max, y_max = convert_box_to_image_coordinates(box_i, H,W)
print("score:", score_i.item()," --- label: ", phrase_i)
display(Image.fromarray(image_source[y_min:y_max, x_min:x_max]))

> *Question 2: Suppose you were using a conventional closed-set object detector trained on a fixed set of categories. Would it be possible to have a detection target of “person” at inference time? Explain why or why not.*

We now refine the detection target by using a more descriptive text prompt that specifies an object together with an associated attribute or context. In this example, the prompt "person with a bike" is used to localise only those people that are semantically associated with a bicycle, rather than all people in the scene.

In [ ]:
image_path = "data/oxbuild_lite/cornmarket_000027.jpg"

TEXT_PROMPT = "person with a bike"
BOX_TRESHOLD = 0.35
TEXT_TRESHOLD = 0.25

image_source, image = load_image(image_path)

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_TRESHOLD,
    text_threshold=TEXT_TRESHOLD,
    device=DEVICE
)

annotated_frame = annotate(image_source=image_source, boxes=boxes, logits=logits, phrases=phrases)

sv.plot_image(annotated_frame, (16, 16))

> *Question 3: Consider a video in which a person initially walks beside a bicycle, then rides it and later leaves it behind. If you use the prompt “person with a bike” as the detection target, how would you expect the detections and the resulting object track to evolve over time? Propose how you might adapt the prompt or detection strategy to maintain a coherent track of the same individual throughout the sequence.*

We now extend open-vocabulary detection from still images to video sequences. Instead of analysing a single image in isolation, the detector is applied independently to each frame of a video as objects and the camera move over time.

In [ ]:
def transform_image(image_pil):
    transform = T.Compose(
        [
            T.RandomResize([800], max_size=1333),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )
    image, _ = transform(image_pil, None)  # 3, h, w
    return image

def run_grounding_dino_on_video(
    video_path,
    model,
    text_prompt,
    box_threshold=0.35,
    text_threshold=0.25,
    transform_fn=None,
    device=DEVICE
):
    """
    Run Grounding DINO frame-by-frame on a video.

    Parameters
    ----------
    video_path : Path to the input video.
    model : Grounding DINO model.
    text_prompt: Text prompt defining the detection target.
    box_threshold: Threshold for filtering bounding boxes.
    text_threshold: Threshold for filtering text–image alignment.
    transform_fn: Function that converts an RGB image into the model input format.
    device: Device the model runs on, "cuda" or "cpu".

    Returns
    -------
    original_frames: Original RGB video frames.
    annotated_frames:  Annotated RGB frames.
    all_boxes: Per-frame bounding boxes.
    all_logits: Per-frame confidence scores.
    all_phrases: Per-frame matched phrases.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open video: {video_path}")

    original_frames = []
    annotated_frames = []
    all_boxes = []
    all_logits = []
    all_phrases = []

    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        original_frames.append(frame_rgb)

        image_source = frame_rgb

        if transform_fn is not None:
            image = transform_fn(Image.fromarray(image_source))
        else:
            image = image_source

        boxes, logits, phrases = predict(
            model=model,
            image=image,
            caption=text_prompt,
            box_threshold=box_threshold,
            text_threshold=text_threshold,
            device=device
        )

        annotated = annotate(
            image_source=image_source,
            boxes=boxes,
            logits=logits,
            phrases=phrases
        )

        annotated_frames.append(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        all_boxes.append(boxes)
        all_logits.append(logits)
        all_phrases.append(phrases)

    cap.release()

    return original_frames, annotated_frames, all_boxes, all_logits, all_phrases

# Re-encode the frames as H.264 so that the notebook can display the video
def save_video_from_frames_h264(frames, output_path, fps=25):
  writer = imageio.get_writer(
      output_path,
      fps=fps,
      codec="libx264",
      quality=8,
      pixelformat="yuv420p"
  )

  for frame_rgb in frames:
      if frame_rgb.dtype != np.uint8:
          frame_rgb = frame_rgb.astype(np.uint8)
      writer.append_data(frame_rgb)

  writer.close()

In [ ]:
VIDEO_PATH = "data/tennis-vest.mp4"
Video(VIDEO_PATH, embed=True)

In [ ]:
orig_frames, ann_frames, boxes, logits, phrases = run_grounding_dino_on_video(
    video_path=VIDEO_PATH,
    model=model,
    text_prompt="person",
    transform_fn=transform_image
)

After running the detector on the video sequence, we visualise the annotated frames. The output shows a single person, in this case a tennis player, correctly detected and annotated in each frame. The bounding boxes illustrate how the detector localises the individual as they move through the scene, producing a sequence of per-frame detections that will be used in the next stage to construct an object track.

In [ ]:
OUT_PATH  = "annotated_video.mp4"
save_video_from_frames_h264(ann_frames, OUT_PATH)
Video(OUT_PATH, embed=True)

## Part II: Object Tracking using IOU
We now move from per-frame **object detection** to **object tracking**, where the goal is to maintain a consistent identity for detected objects as they move across consecutive video frames. In this practical, tracking is done through tracking by detection: objects are detected independently in each frame and then associated over time.

To associate detections between frames, we use **Intersection over Union (IoU)** as a measure of spatial overlap between bounding boxes. The underlying assumption is that an object's position changes smoothly over time, so the bounding box corresponding to the same object in consecutive frames will have a high overlap. By matching detections to existing tracks based on IoU, we can link frame-wise detections into continuous object trajectories.

In this section, you will implement a **simple IoU-based tracker**. Starting from the bounding boxes produced by Grounding DINO, you will convert detections into image coordinates, compute IoU between bounding boxes across frames and assign detections to tracks using a threshold on IoU.
### Learning outcomes

* Understand basic object tracking principles: explain how detections can be linked across frames to form object tracks using spatial overlap.
* Apply IoU-based tracking: use an Intersection-over-Union (IoU) criterion to associate detections over time and maintain simple object identities.
* Interpret tracking parameters: understand the effect of the IoU threshold.
* Analyse tracking results: evaluate the quality of generated tracks and identify common failure cases.
* Relate tracking to application goals: reason about which objects and tracking outputs are useful for analysing activities in a sports scenario (e.g. player movement in tennis).
* Recognise limitations of simple trackers: identify scenarios where IoU-based tracking is insufficient and motivate the need for more advanced tracking methods.

We use the tennis video with the prompt "person" which provides a controlled, single-object setting for introducing tracking.
This sequence of per-frame bounding boxes will serve as the input to the IoU-based tracker in the next section, where we will associate detections across frames to construct a continuous track.

In [ ]:
VIDEO_PATH = "data/tennis-vest.mp4"

orig_frames, ann_frames, boxes, logits, phrases = run_grounding_dino_on_video(
    video_path=VIDEO_PATH,
    model=model,
    text_prompt="person",
    transform_fn=transform_image
)

In [ ]:
OUT_PATH  = "annotated_video.mp4"
save_video_from_frames_h264(ann_frames, OUT_PATH)
Video(OUT_PATH, embed=True)

### Example: IoU matching from frame t to frame t+1


1) Pick two consecutive frames and convert their detections to pixel boxes

In [ ]:
t = 0  # change t to different frame needed
H, W, _ = orig_frames[t].shape

def dino_box_to_xyxy(box, H, W):
    # box: tensor or array [x_c, y_c, w, h] in [0,1]
    if hasattr(box, "detach"):
        box = box.detach().cpu().numpy()
    x_c, y_c, bw, bh = box

    x_c *= W; y_c *= H; bw *= W; bh *= H

    x1 = x_c - bw / 2
    y1 = y_c - bh / 2
    x2 = x_c + bw / 2
    y2 = y_c + bh / 2

    x1 = max(0, min(W - 1, x1))
    y1 = max(0, min(H - 1, y1))
    x2 = max(0, min(W - 1, x2))
    y2 = max(0, min(H - 1, y2))
    return np.array([x1, y1, x2, y2], dtype=np.float32)

dets_t   = [dino_box_to_xyxy(b, H, W) for b in boxes[t]]
dets_t1  = [dino_box_to_xyxy(b, H, W) for b in boxes[t+1]]

print("Frame", t,   "num dets:", len(dets_t))
print("Frame", t+1, "num dets:", len(dets_t1))

2) Compute the IoU matrix


    Rows are detections in frame t, columns are detections in frame t+1.

In [ ]:
iou_mat = np.zeros((len(dets_t), len(dets_t1)), dtype=np.float32)

def iou_xyxy(a, b):
    # a, b: [x1,y1,x2,y2]
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter + 1e-6
    return inter / union

for i, a in enumerate(dets_t):
    for j, b in enumerate(dets_t1):
        iou_mat[i, j] = iou_xyxy(a, b)

iou_mat

3) Choose the best match using IoU

   For each detection in frame t, pick the detection in t+1 with the highest IoU.

In [ ]:
best_j_for_i = iou_mat.argmax(axis=1) if iou_mat.size else []
best_iou_for_i = iou_mat.max(axis=1) if iou_mat.size else []

for i in range(len(dets_t)):
    j = int(best_j_for_i[i])
    print(f"Det {i} in frame {t} best matches det {j} in frame {t+1} with IoU = {best_iou_for_i[i]:.3f}")

4) Apply a threshold to decide if it is the same object

In [ ]:
IOU_THRESHOLD = 0.1  # for your tennis video this is usually sufficient

matches = []
for i in range(len(dets_t)):
    j = int(best_j_for_i[i])
    if best_iou_for_i[i] >= IOU_THRESHOLD:
        matches.append((i, j, float(best_iou_for_i[i])))

print("Matches:", matches)

5) Visualise the matched boxes on the two frames

In [ ]:
def draw_box(frame_rgb, box_xyxy, label):
    x1, y1, x2, y2 = map(int, box_xyxy)
    out = frame_rgb.copy()
    cv2.rectangle(out, (x1,y1), (x2,y2), (0,255,0), 2)
    cv2.putText(out, label, (x1, max(0, y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
    return out

if matches:
    for ii, mat in enumerate(matches):
      i, j, val = mat
      print(f"MATCHES {ii}")
      f0 = draw_box(orig_frames[t],   dets_t[i],  f"t={t}, det={i}")
      f1 = draw_box(orig_frames[t+1], dets_t1[j], f"t={t+1}, det={j}, IoU={val:.2f}")

      plt.figure(figsize=(12,6))
      plt.subplot(1,2,1); plt.imshow(f0); plt.axis("off")
      plt.subplot(1,2,2); plt.imshow(f1); plt.axis("off")
      plt.show()
else:
    print("No matches found above threshold; try lowering IOU_THRESHOLD or picking a different t.")

> Task: Repeat the above section using "shoe" as a prompt and understand how the code works when there are multiple objects detected. Test with different frame index (by changing `t`). For example try with the prompt "shoe" and look at when `t=65`, do you notice any issues with the matches?

> *Question 4: Consider two bounding boxes in consecutive video frames. What geometric information does Intersection over Union (IoU) capture about their spatial relationship, and why is IoU an appropriate measure for deciding whether the two boxes correspond to the same moving object? Discuss how IoU would change if the object moves slowly versus rapidly between frames.*

> *Question 5: The IoU-based matching example assumes that any detection in frame t can be matched to any detection in frame t+1. Under what conditions is this assumption reasonable and why does it hold in the above cases? How would this assumption break down if the detector were producing detections for multiple object categories or multiple instances, and what additional constraints would be needed to maintain correct tracking?*

We now link object detections across consecutive frames to form simple tracks. This is done using a lightweight **Intersection-over-Union (IoU)-based tracking approach**. At each frame, detections are matched to existing tracks by comparing their bounding boxes, detections with sufficiently high IoU are assigned to the same track. Tracks that are not matched for several consecutive frames are terminated, while unmatched detections start new tracks.

In [ ]:
def track_iou(
    orig_frames,
    boxes_by_frame,
    iou_threshold=0.3,
    max_missed=5
):
    """
    Returns
    -------
    tracks: dict track_id -> list of (frame_idx, xyxy_box)
    """
    H, W, _ = orig_frames[0].shape

    next_track_id = 0
    active = {}   # track_id -> {"box": xyxy, "missed": int}
    tracks = {}   # track_id -> [(t, xyxy), ...]

    for t in range(len(orig_frames)):
        dets = [dino_box_to_xyxy(b, H, W) for b in boxes_by_frame[t]]

        # Build IoU candidates: (iou, track_id, det_id)
        candidates = []
        for tid, st in active.items():
            for j, d in enumerate(dets):
                candidates.append((iou_xyxy(st["box"], d), tid, j))
        candidates.sort(reverse=True, key=lambda x: x[0])

        matched_tracks = set()
        matched_dets = set()

        # Greedy matching by highest IoU
        for iou_val, tid, j in candidates:
            if iou_val < iou_threshold:
                break
            if tid in matched_tracks or j in matched_dets:
                continue
            matched_tracks.add(tid)
            matched_dets.add(j)

            active[tid]["box"] = dets[j]
            active[tid]["missed"] = 0
            tracks.setdefault(tid, []).append((t, dets[j]))

        # Unmatched tracks: increment missed
        for tid in list(active.keys()):
            if tid not in matched_tracks:
                active[tid]["missed"] += 1
                if active[tid]["missed"] > max_missed:
                    del active[tid]

        # Unmatched detections: start new tracks
        for j, d in enumerate(dets):
            if j in matched_dets:
                continue
            tid = next_track_id
            next_track_id += 1
            active[tid] = {"box": d, "missed": 0}
            tracks.setdefault(tid, []).append((t, d))

    return tracks

tracks = track_iou(
    orig_frames=orig_frames,
    boxes_by_frame=boxes,
    iou_threshold=0.1,
    max_missed=5
)

len(tracks), list(tracks.keys())[:5]

We now visualise the result of the IoU-based tracking by overlaying the computed track identities onto the original video frames. Each tracked object is drawn with a consistent bounding box and identifier across frames, allowing us to directly observe whether the tracker has successfully maintained object identity over time.

In [ ]:
def draw_tracks_on_frames(frames_rgb, tracks):
    frames_out = [f.copy() for f in frames_rgb]

    for tid, elems in tracks.items():
        for (t, box) in elems:
            x1, y1, x2, y2 = box.astype(int)
            cv2.rectangle(frames_out[t], (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frames_out[t], f"ID {tid}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return frames_out


In [ ]:
tracked_frames = draw_tracks_on_frames(orig_frames, tracks)

OUT_PATH  = "tracked_video.mp4"
save_video_from_frames_h264(tracked_frames, OUT_PATH)
Video(OUT_PATH, embed=True)

> *Task: Repeat the above tracking with different objects of interest. Consider what may be useful for tracking in a tennis game to analyse tennis players style such as the hand, racket, etc.*

> *Question 6: Identify potential failure modes of the IoU-based tracking approach used here.*

> Question 7: How could you improve the tracking to fix such failure cases?

Repeat the above tracking with the video
```
VIDEO_PATH = "data/horsejump-stick.mp4"
```

Consider tracking "horse" or "person".



Examine how the tracked bounding boxes move over time. Consider what information about motion and dynamics can be extracted from these tracks (e.g. trajectories, speed, acceleration) and reflect on the limitations of bounding-box tracking for capturing fine-grained motion. How might tracking points or keypoints provide a more detailed description of object motion in such a scenario?

## References

* Liu, Shilong, et al. "Grounding dino: Marrying dino with grounded pre-training for open-set object detection." European conference on computer vision. Cham: Springer Nature Switzerland, 2024. [link](https://arxiv.org/pdf/2303.05499)
* Video from [DAVIS: Densely Annotated VIdeo Segmentation Dataset](https://davischallenge.org/)
## Code Borrowed and adapted from the following sources

* https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/zero-shot-object-detection-with-grounding-dino.ipynb
